# Ginzburg-Landau oscillator: phase reduction

The complex Ginzburg-Landau oscillator in real coordinates (Wilson & Ermentrout,
PRL 2019). Uncoupled, its limit cycle is (cos qt, sin qt) with period 2π/q, and its adjoint
(phase response) and isostable functions are known exactly — a good test of XPP 8.1's
phase-reduction tools.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.1 and its AUTO through pyauto-xpp.*

In [1]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [2]:
mb = (ModelBuilder("ginzburg_landau")
      .ode("x", "x*(1-x^2-y^2)-q*(x^2+y^2)*y")
      .ode("y", "y*(1-x^2-y^2)+q*(x^2+y^2)*x")
      .parameter(q=1, d=0.9)
      .initial(x=1, y=0)
      .option(total=40, dt=0.01, meth="rk4"))
print(mb.render())

x'=x*(1-x^2-y^2)-q*(x^2+y^2)*y
y'=y*(1-x^2-y^2)+q*(x^2+y^2)*x
par q=1,d=0.9
init x=1,y=0
@ total=40,dt=0.01,meth=rk4
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [3]:
ex = Explorer(mb, kind="plane", x="x", y="y", window=(-1.5, 1.5, -1.5, 1.5),
              sliders={"q": (0.5, 2)}, title="Ginzburg-Landau oscillator")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**One period, the adjoint, and H for the paper's coupling.** XPP 8.1: Initialconds >
Periodic, then nUmerics > Averaging. The coupling of a pair of cells is
`(x'-x) - d (y'-y)` and `(y'-y) + d (x'-x)`.

In [4]:
import numpy as np
m = ex.m
m.run(total=50, trans=0)
m.buttons.initialconds.periodic()
T = m.numerics["total"]
print(f"period {T:.6f}  (exact 2*pi/q = {2*np.pi/m.pars['q']:.6f})")
A = m.buttons.numerics.averaging
A.new_adj(); adj = m.data
A.make_h("(x'-x)-d*(y'-y)", "(y'-y)+d*(x'-x)"); H = m.data
fig = go.Figure([go.Scatter(x=adj.t, y=adj["x"], name="adjoint x"), go.Scatter(x=adj.t, y=adj["y"], name="adjoint y"),
                 go.Scatter(x=H.t, y=H["x"], name="H")])
show_figure(fig, xaxis_title="phase (time along the orbit)", height=320)
A.orbit()

period 6.283185  (exact 2*pi/q = 6.283185)


'menu    Adjoint -> (O)rbit\n'

**Second order (H2) and gamma**, also from XPP 8.1.

In [5]:
n = len(m.log)
A.h2()
print("\n".join(l.strip() for l in m.log[n:].strip().splitlines()))

H2 computed (two coupled cells, second order): H2(0)=0, max |H2|=2.1105549
g2+g3 = -2 odd(H2); gamma = (g2'(0)+g3'(0))/4 = -0.81 (from A: -0.81, from Bf: -0.81)


## Things to try

- Change `q`: the period follows 2π/q exactly.